# Эксперимент 14: MRL + No-Projection GAT

Анализ результатов:
1. **Статистика MRL-графа** — что дала фильтрация (IDF + min-degree + per-cell)
2. **Сравнение с v3_unified** (exp 08) — на сколько меньше рёбер/токенов
3. **Кривые обучения** GAT без входных проекций
4. **Threshold-based evaluation** — F1/P/R + ROC-AUC/AP на val/test/cross-domain
5. **Сравнение метрик с v3 baseline** (exp 09/11)
6. **Ablation: BCE vs NT-Xent** на v14 архитектуре
7. **t-SNE эмбеддингов** (опционально)

Архитектура:
- `col_dim = row_dim = token_dim = hidden_dim = edge_dim = output_dim = 312`
- `use_input_projection=False` — учатся только GAT-слои + output_head
- column embeddings обрезаны 4096 → 312 через MRL (Matryoshka)

Переключатель `LOSS` ниже определяет, какая модель используется в основном анализе (разделы 3-5).
Раздел 6 грузит обе и сравнивает.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
MRL_DIR = DATA_DIR / "graphs" / "v14_mrl"
MRL_CROSS_DIR = DATA_DIR / "graphs" / "v14_mrl_cross"
V3_DIR = DATA_DIR / "graphs" / "v3_unified"

# Активный лосс для разделов 3-5 (раздел 6 грузит оба для ablation)
LOSS = "bce"   # "bce" или "ntxent"


def model_paths(loss: str) -> tuple[Path, Path, Path]:
    """Пути к (state_dict, config, history) для выбранного loss'а."""
    suffix = "_bce" if loss == "bce" else ""
    base = OUTPUT_DIR / f"v14_mrl_gat{suffix}_model"
    return (base.with_suffix(".pt"),
            base.with_suffix(".config.json"),
            base.with_suffix(".history.json"))


MODEL_PATH, MODEL_CONFIG_PATH, HISTORY_PATH = model_paths(LOSS)
print(f"Активная модель: LOSS={LOSS!r} → {MODEL_PATH.name}")

---
## 1. Статистика MRL-графа (exp 14 build)

In [ ]:
with open(MRL_DIR / "stats.json") as f:
    stats = json.load(f)

print(f"Датасетов:        {stats['n_datasets']}")
print(f"Строк (row):      {stats['n_rows']:,}")
print(f"Токенов (token):  {stats['n_tokens']:,}")
print(f"Рёбер:            {stats['n_edges']:,}")
print(f"col_dim:          {stats['col_dim']} (MRL target_dim={stats['mrl_target_dim']})")
print(f"max_tokens/cell:  {stats['max_tokens_per_cell']}")
print(f"min_token_count:  {stats['min_token_count']}")
print()
print(f"Labeled pairs:    {stats['n_labeled']:,}")
print(f"  train:          {stats['n_train']:,} ({100*stats['n_train']/stats['n_labeled']:.1f}%)")
print(f"  val:            {stats['n_val']:,} ({100*stats['n_val']/stats['n_labeled']:.1f}%)")
print(f"  test:           {stats['n_test']:,} ({100*stats['n_test']/stats['n_labeled']:.1f}%)")

In [ ]:
fs = stats["filter_stats"]
raw_e = fs["raw_edges"]
raw_t = fs["raw_tokens"]

print("Этапы фильтрации токенов:")
print(f"  Сырых токенов:               {raw_t:,}")
print(f"  Удалено IDF (df > {fs['idf_threshold']:.0%}):   {fs['tokens_removed_idf']:,} ({100*fs['tokens_removed_idf']/raw_t:.1f}%)")
print(f"  Удалено min-deg (df < {fs['min_token_count']}):   {fs['tokens_removed_mindeg']:,} ({100*fs['tokens_removed_mindeg']/raw_t:.1f}%)")
print(f"  Финальных токенов:           {fs['final_tokens']:,}")
print()
print("Этапы фильтрации рёбер:")
print(f"  Сырых рёбер:                 {raw_e:,}")
print(f"  Удалено IDF:                 {fs['edges_removed_idf']:,} ({100*fs['edges_removed_idf']/raw_e:.1f}%)")
print(f"  Удалено min-deg:             {fs['edges_removed_mindeg']:,} ({100*fs['edges_removed_mindeg']/raw_e:.1f}%)")
print(f"  Удалено per-cell limit:      {fs['edges_removed_cell_limit']:,} ({100*fs['edges_removed_cell_limit']/raw_e:.1f}%)")
print(f"  Финальных рёбер:             {fs['final_edges']:,} ({100*fs['final_edges']/raw_e:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
stages = ["Raw", "-IDF", "-MinDeg", "-PerCell\n(final)"]
values = [
    raw_e,
    raw_e - fs["edges_removed_idf"],
    raw_e - fs["edges_removed_idf"] - fs["edges_removed_mindeg"],
    fs["final_edges"],
]
colors = ["#9E9E9E", "#FF9800", "#F44336", "#4CAF50"]
bars = ax.bar(stages, values, color=colors)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{v:,}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Число рёбер")
ax.set_title("Waterfall: эффект каждого этапа фильтрации")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

---
## 2. Сравнение с v3_unified (baseline без MRL и min-degree)

In [ ]:
v3_stats_path = V3_DIR / "stats.json"
if v3_stats_path.exists():
    with open(v3_stats_path) as f:
        v3_stats = json.load(f)

    print(f"{'Метрика':<25} {'v3_unified':>15} {'v14_mrl':>15} {'Ratio':>10}")
    print("-" * 70)
    for key, label in [("n_rows", "Строк"), ("n_tokens", "Токенов"), ("n_edges", "Рёбер")]:
        v3, v14 = v3_stats[key], stats[key]
        print(f"{label:<25} {v3:>15,} {v14:>15,} {v14/v3:>9.2%}")
    print(f"{'col_dim (col emb)':<25} {'4096':>15} {stats['col_dim']:>15} {stats['col_dim']/4096:>9.2%}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, title in zip(
        axes,
        ["n_rows", "n_tokens", "n_edges"],
        ["Строк", "Уникальных токенов", "Рёбер token→row"],
    ):
        vals = [v3_stats[key], stats[key]]
        bars = ax.bar(["v3_unified", "v14_mrl"], vals, color=["#9E9E9E", "#4CAF50"])
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{v:,}", ha="center", va="bottom", fontweight="bold")
        ax.set_title(title)
        ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.tight_layout()
    plt.show()
else:
    print("v3_unified/stats.json не найден — пропуск сравнения")

---
## 3. Кривые обучения GAT (no projection)

Активный loss: `LOSS` из ячейки конфигурации.

In [ ]:
with open(HISTORY_PATH) as f:
    history = json.load(f)

epochs = list(range(1, len(history["train_loss"]) + 1))
print(f"Loss type: {LOSS}")
print(f"Эпох обучения: {len(epochs)}")
print(f"Финальный train_loss: {history['train_loss'][-1]:.4f}")
if history.get("val_loss"):
    best_val_idx = int(np.argmin(history["val_loss"]))
    print(f"Лучший val_loss:     {history['val_loss'][best_val_idx]:.4f} (эпоха {best_val_idx + 1})")
if history.get("lr"):
    print(f"Финальный LR:        {history['lr'][-1]:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(epochs, history["train_loss"], label="Train Loss", color="#2196F3", linewidth=1.5)
if history.get("val_loss"):
    val_epochs = list(range(1, len(history["val_loss"]) + 1))
    ax.plot(val_epochs, history["val_loss"], label="Val Loss", color="#F44336", linewidth=1.5)
    best = int(np.argmin(history["val_loss"]))
    ax.axvline(best + 1, color="green", linestyle="--", alpha=0.7, label=f"Best @ ep {best+1}")
    ax.scatter([best+1], [history["val_loss"][best]], color="green", zorder=5)
ax.set_xlabel("Эпоха"); ax.set_ylabel(f"{LOSS.upper()} Loss")
ax.set_title(f"Кривые обучения ({LOSS})"); ax.legend()

ax = axes[1]
if history.get("lr"):
    ax.plot(epochs, history["lr"], color="#9C27B0", linewidth=1.5)
    ax.set_xlabel("Эпоха"); ax.set_ylabel("Learning Rate")
    ax.set_title("LR schedule (warmup + cosine)")
    ax.set_yscale("log")

plt.tight_layout()
plt.show()

---
## 4. Threshold-based evaluation (val → θ → test/cross)

In [ ]:
from table_unifier.evaluation.clustering import (
    evaluate_pairs_at_threshold,
    evaluate_pairs_auc,
    find_best_threshold,
)
from table_unifier.models.entity_resolution import EntityResolutionGAT, PairClassifier
from table_unifier.training.er_trainer import get_row_embeddings

device = "cuda" if torch.cuda.is_available() else "cpu"


def build_backbone(cfg: dict) -> EntityResolutionGAT:
    return EntityResolutionGAT(
        row_dim=cfg["row_dim"], token_dim=cfg["token_dim"], col_dim=cfg["col_dim"],
        hidden_dim=cfg["hidden_dim"], edge_dim=cfg["edge_dim"], output_dim=cfg["output_dim"],
        num_gnn_layers=cfg["num_gnn_layers"], num_heads=cfg["num_heads"],
        dropout=cfg["dropout"], attention_dropout=cfg["attention_dropout"],
        bidirectional=cfg["bidirectional"],
        use_input_projection=cfg["use_input_projection"],
    )


def load_backbone(model_path: Path, cfg: dict, loss: str) -> EntityResolutionGAT:
    """BCE сохраняется как PairClassifier, NT-Xent — как чистый backbone."""
    backbone = build_backbone(cfg)
    state = torch.load(model_path, map_location=device, weights_only=True)
    if loss == "bce":
        wrapper = PairClassifier(backbone, embedding_dim=cfg["output_dim"])
        wrapper.load_state_dict(state)
        wrapper.to(device).eval()
        return wrapper.backbone
    backbone.load_state_dict(state)
    backbone.to(device).eval()
    return backbone


def evaluate_loss(loss: str) -> dict:
    """Полная оценка модели на in-domain test + cross-domain. Возвращает dict."""
    mp, cp, _ = model_paths(loss)
    if not mp.exists():
        print(f"  [skip] {mp.name} не найден")
        return {}
    with open(cp) as f:
        cfg = json.load(f)

    graph = torch.load(MRL_DIR / "graph.pt", weights_only=False)
    val_pairs = torch.load(MRL_DIR / "val_pairs.pt", weights_only=False)
    test_pairs = torch.load(MRL_DIR / "test_pairs.pt", weights_only=False)

    backbone = load_backbone(mp, cfg, loss)
    embeddings = get_row_embeddings(backbone, graph, device="cpu")

    threshold, _ = find_best_threshold(embeddings, val_pairs)
    out = {"threshold": threshold, "loss": loss}
    for split_name, pairs in [("val", val_pairs), ("test", test_pairs)]:
        m = evaluate_pairs_at_threshold(embeddings, pairs, threshold)
        m.update(evaluate_pairs_auc(embeddings, pairs))
        out[split_name] = m
    del graph, embeddings
    torch.cuda.empty_cache()

    cross = []
    if MRL_CROSS_DIR.exists():
        for ds_dir in sorted(MRL_CROSS_DIR.iterdir()):
            if not ds_dir.is_dir() or not (ds_dir / "graph.pt").exists():
                continue
            cg = torch.load(ds_dir / "graph.pt", weights_only=False)
            bb = load_backbone(mp, cfg, loss)
            emb = get_row_embeddings(bb, cg, device="cpu")
            cd = {"name": ds_dir.name}
            lp_path = ds_dir / "labeled_pairs.pt"
            if lp_path.exists():
                lp = torch.load(lp_path, weights_only=False)
                metrics = evaluate_pairs_at_threshold(emb, lp, threshold)
                metrics.update(evaluate_pairs_auc(emb, lp))
                cd.update(metrics)
            cross.append(cd)
            del cg, emb
            torch.cuda.empty_cache()
    out["cross_domain"] = cross
    return out


results = evaluate_loss(LOSS)
best_threshold = results["threshold"]
print(f"Optimal θ = {best_threshold:.4f}")
for split_name in ("val", "test"):
    print(f"\n{split_name.upper()}:")
    for k, v in results[split_name].items():
        if isinstance(v, float):
            print(f"  {k:20s}: {v:.4f}")
        else:
            print(f"  {k:20s}: {v}")

for cd in results["cross_domain"]:
    print(f"  cross/{cd['name']:15s}  F1={cd.get('f1', 0):.4f}  AUC={cd.get('roc_auc', 0):.4f}")

with open(OUTPUT_DIR / f"v14_evaluation_results_{LOSS}.json", "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

In [ ]:
# Bar chart in-domain TEST
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

test_m = results["test"]
ax = axes[0]
names = ["F1", "Precision", "Recall"]
vals = [test_m.get("f1", 0), test_m.get("precision", 0), test_m.get("recall", 0)]
bars = ax.bar(names, vals, color=["#4CAF50", "#2196F3", "#FF9800"])
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{v:.3f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, 1.05); ax.set_title(f"In-domain TEST ({LOSS}) @ θ={best_threshold:.3f}")

ax = axes[1]
names = ["ROC-AUC", "Avg Precision"]
vals = [test_m.get("roc_auc", 0), test_m.get("avg_precision", 0)]
bars = ax.bar(names, vals, color=["#9C27B0", "#00BCD4"])
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{v:.3f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, 1.05); ax.set_title("In-domain TEST: ranking metrics (threshold-free)")

plt.tight_layout()
plt.show()

In [ ]:
# Cross-domain bar chart
cross_results = results["cross_domain"]
if cross_results:
    names = [r["name"] for r in cross_results]
    f1s = [r.get("f1", 0) for r in cross_results]
    precs = [r.get("precision", 0) for r in cross_results]
    recs = [r.get("recall", 0) for r in cross_results]
    aucs = [r.get("roc_auc", 0) for r in cross_results]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(names))
    w = 0.25

    ax = axes[0]
    ax.bar(x - w, f1s, w, label="F1", color="#4CAF50")
    ax.bar(x, precs, w, label="Precision", color="#2196F3")
    ax.bar(x + w, recs, w, label="Recall", color="#FF9800")
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylim(0, 1.05); ax.legend()
    ax.set_title(f"Cross-domain ({LOSS}) @ θ={best_threshold:.3f}")

    ax = axes[1]
    ax.bar(names, aucs, color="#9C27B0")
    for i, v in enumerate(aucs):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontweight="bold")
    ax.set_ylim(0, 1.05); ax.set_title("Cross-domain ROC-AUC")

    plt.tight_layout()
    plt.show()

---
## 5. Сравнение с v3 baseline (тот же loss, c проекциями vs MRL no-proj)

In [ ]:
import pandas as pd

v3_suffix = "_bce" if LOSS == "bce" else ""
v3_results_path = OUTPUT_DIR / f"v3_evaluation_results{v3_suffix}.json"

if v3_results_path.exists():
    with open(v3_results_path) as f:
        v3_results = json.load(f)

    rows = []
    for label, res in [(f"v3 GAT {LOSS} (proj→128)", v3_results),
                       (f"v14 GAT {LOSS} (MRL no-proj 312)", results)]:
        t = res.get("test", {})
        rows.append({
            "Model": label,
            "Threshold": res.get("threshold"),
            "F1": t.get("f1"),
            "Precision": t.get("precision"),
            "Recall": t.get("recall"),
            "ROC-AUC": t.get("roc_auc"),
            "AP": t.get("avg_precision"),
        })
    df_test = pd.DataFrame(rows)
    print("In-domain TEST:")
    print(df_test.to_string(index=False))

    cross_v3 = {r["name"]: r for r in v3_results.get("cross_domain", [])}
    cross_v14 = {r["name"]: r for r in results.get("cross_domain", [])}
    all_names = sorted(set(cross_v3) | set(cross_v14))

    if all_names:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        x = np.arange(len(all_names))
        w = 0.35

        ax = axes[0]
        v3_f1 = [cross_v3.get(n, {}).get("f1", 0) for n in all_names]
        v14_f1 = [cross_v14.get(n, {}).get("f1", 0) for n in all_names]
        ax.bar(x - w/2, v3_f1, w, label="v3", color="#9E9E9E")
        ax.bar(x + w/2, v14_f1, w, label="v14 MRL", color="#4CAF50")
        ax.set_xticks(x); ax.set_xticklabels(all_names)
        ax.set_title(f"Cross-domain F1 ({LOSS})"); ax.set_ylim(0, 1.05); ax.legend()

        ax = axes[1]
        v3_auc = [cross_v3.get(n, {}).get("roc_auc", 0) for n in all_names]
        v14_auc = [cross_v14.get(n, {}).get("roc_auc", 0) for n in all_names]
        ax.bar(x - w/2, v3_auc, w, label="v3", color="#9E9E9E")
        ax.bar(x + w/2, v14_auc, w, label="v14 MRL", color="#4CAF50")
        ax.set_xticks(x); ax.set_xticklabels(all_names)
        ax.set_title(f"Cross-domain ROC-AUC ({LOSS})"); ax.set_ylim(0, 1.05); ax.legend()

        plt.tight_layout()
        plt.show()
else:
    print(f"v3 baseline ({v3_results_path.name}) не найден — пропуск сравнения.")

---
## 6. Ablation: BCE vs NT-Xent (на v14 MRL архитектуре)

Грузим обе модели и сравниваем при одинаковом графе/спите/архитектуре — изолируем эффект loss'а.

In [ ]:
# Грузим оба результата (если они уже посчитаны/сохранены текущей сессией) или считаем заново
ablation: dict[str, dict] = {}
for loss in ("bce", "ntxent"):
    cached = OUTPUT_DIR / f"v14_evaluation_results_{loss}.json"
    if cached.exists() and loss != LOSS:
        with open(cached) as f:
            ablation[loss] = json.load(f)
        print(f"  [{loss}] загружено из {cached.name}")
    elif loss == LOSS:
        ablation[loss] = results
    else:
        print(f"  [{loss}] считаю...")
        r = evaluate_loss(loss)
        if r:
            ablation[loss] = r
            with open(cached, "w") as f:
                json.dump(r, f, ensure_ascii=False, indent=2)

print("\nДоступно:", list(ablation.keys()))

In [ ]:
# Сводная таблица
if len(ablation) >= 1:
    rows = []
    for loss, res in ablation.items():
        t = res.get("test", {})
        rows.append({
            "Loss": loss,
            "Threshold": res.get("threshold"),
            "F1": t.get("f1"),
            "Precision": t.get("precision"),
            "Recall": t.get("recall"),
            "ROC-AUC": t.get("roc_auc"),
            "AP": t.get("avg_precision"),
        })
    print("v14 MRL — In-domain TEST:")
    print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Сравнение кривых обучения BCE vs NT-Xent (на разных шкалах — два subplot'а)
histories = {}
for loss in ("bce", "ntxent"):
    _, _, hp = model_paths(loss)
    if hp.exists():
        with open(hp) as f:
            histories[loss] = json.load(f)

if len(histories) >= 1:
    fig, axes = plt.subplots(1, len(histories), figsize=(7 * len(histories), 5), squeeze=False)
    for ax, (loss, h) in zip(axes[0], histories.items()):
        ep = list(range(1, len(h["train_loss"]) + 1))
        ax.plot(ep, h["train_loss"], label="train", color="#2196F3")
        if h.get("val_loss"):
            v_ep = list(range(1, len(h["val_loss"]) + 1))
            ax.plot(v_ep, h["val_loss"], label="val", color="#F44336")
        ax.set_title(f"{loss}"); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Cross-domain ablation
if len(ablation) >= 2:
    cd_per_loss = {loss: {r["name"]: r for r in res.get("cross_domain", [])}
                   for loss, res in ablation.items()}
    all_names = sorted({n for loss_map in cd_per_loss.values() for n in loss_map})

    if all_names:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        x = np.arange(len(all_names))
        w = 0.35
        colors = {"bce": "#FF9800", "ntxent": "#2196F3"}

        ax = axes[0]
        for i, loss in enumerate(ablation.keys()):
            f1s = [cd_per_loss[loss].get(n, {}).get("f1", 0) for n in all_names]
            offset = (i - 0.5) * w
            ax.bar(x + offset, f1s, w, label=loss, color=colors.get(loss))
        ax.set_xticks(x); ax.set_xticklabels(all_names)
        ax.set_title("v14 Cross-domain F1: BCE vs NT-Xent")
        ax.set_ylim(0, 1.05); ax.legend()

        ax = axes[1]
        for i, loss in enumerate(ablation.keys()):
            aucs = [cd_per_loss[loss].get(n, {}).get("roc_auc", 0) for n in all_names]
            offset = (i - 0.5) * w
            ax.bar(x + offset, aucs, w, label=loss, color=colors.get(loss))
        ax.set_xticks(x); ax.set_xticklabels(all_names)
        ax.set_title("v14 Cross-domain ROC-AUC: BCE vs NT-Xent")
        ax.set_ylim(0, 1.05); ax.legend()

        plt.tight_layout()
        plt.show()

---
## 7. t-SNE эмбеддингов (cross-domain)

In [ ]:
from sklearn.manifold import TSNE

with open(MODEL_CONFIG_PATH) as f:
    cfg_active = json.load(f)

if MRL_CROSS_DIR.exists():
    cross_dirs = sorted([d for d in MRL_CROSS_DIR.iterdir()
                         if d.is_dir() and (d / "graph.pt").exists()])
    if cross_dirs:
        n = len(cross_dirs)
        fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
        if n == 1:
            axes = [axes]

        for ax, ds_dir in zip(axes, cross_dirs):
            cg = torch.load(ds_dir / "graph.pt", weights_only=False)
            bb = load_backbone(MODEL_PATH, cfg_active, LOSS)
            emb = get_row_embeddings(bb, cg, device="cpu").numpy()

            n_sub = min(2000, len(emb))
            idx = np.random.choice(len(emb), n_sub, replace=False)
            emb_sub = emb[idx]

            tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
            coords = tsne.fit_transform(emb_sub)

            ax.scatter(coords[:, 0], coords[:, 1], s=3, alpha=0.5)
            ax.set_title(f"{ds_dir.name} ({n_sub} rows, {LOSS})")
            ax.set_xticks([]); ax.set_yticks([])

            del cg, emb
            torch.cuda.empty_cache()

        plt.tight_layout()
        plt.show()